# Amparo -- M2: Evaluacion del modelo fine-tuneado

Este notebook corre el harness de evaluacion de M2 sobre el adaptador LoRA
que entreno M1 (`baseline_finetune.ipynb`, ya publicado en la wiki). No
modifica ni depende de que M1 haya dejado archivos guardados: vuelve a
generar las respuestas baseline y fine-tuned sobre el **mismo split de
validacion** (mismo seed) para que los numeros sean comparables con el
baseline ya publicado (3.4% / 17.7% de similitud lexica).

Fases: generar respuestas (baseline y fine-tuned) -> liberar el adaptador y
reusar el modelo base como juez -> LLM-as-judge + sondeo de position bias ->
metricas clasicas + metrica de dominio juridico (sin GPU) -> sesgos de
longitud y auto-preferencia -> scorecard, persistido en Drive.

Toda la logica pesada vive en `tools/evaluation/` (repo principal, no en
este notebook) -- este notebook solo clona el repo, instala dependencias, y
orquesta las llamadas.

Antes de correr: `Entorno de ejecucion > Cambiar tipo de entorno de
ejecucion > GPU`.


In [6]:
# Clona el repo (o actualiza si ya existe de una corrida anterior en esta VM)
import os

if not os.path.isdir("Amparo"):
    !git clone https://github.com/TomasPosada0626/Amparo.git
else:
    !cd Amparo && git pull

%cd Amparo


Cloning into 'Amparo'...
remote: Enumerating objects: 144, done.
remote: Counting objects: 100% (144/144), done.
remote: Compressing objects: 100% (83/83), done.
remote: Total 144 (delta 60), reused 136 (delta 55), pack-reused 0 (from 0)
Receiving objects: 100% (144/144), 295.22 KiB | 1.28 MiB/s, done.
Resolving deltas: 100% (60/60), done.
/content/Amparo/Amparo


In [7]:
# Dependencias puras de tools/evaluation (metricas clasicas, tests)
!pip install -q -r requirements.txt


In [8]:
# Stack de ML pesado -- igual que en M1 (baseline_finetune.ipynb, celda 1):
# se instala aqui y NO en requirements.txt del repo, para no arriesgar
# reemplazar el build de PyTorch con CUDA que Colab ya trae preinstalado.
# bert-score va aqui tambien (no en requirements.txt) porque depende de
# torch de forma transitiva.
!pip install -q -U transformers peft bitsandbytes accelerate bert-score
!pip install -U "bitsandbytes>=0.46.1"


## Configuracion

Las constantes (seed, val_fraction, modelo base, rutas de Drive) viven en
`tools/evaluation/config.py` -- deben coincidir con `RANDOM_SEED=42` y
`VAL_FRACTION=0.15` de M1 para evaluar sobre el mismo split.


In [9]:
from google.colab import drive

drive.mount('/content/drive')

from tools.evaluation import config, dataset

records = dataset.load_records()
train_records, val_records = dataset.stratified_split(records)
system_prompt = dataset.system_prompt(records)

print(f"Total: {len(records)} | train: {len(train_records)} | val: {len(val_records)}")
assert len(val_records) == 201, "El split no coincide con el de M1 -- revisar RANDOM_SEED/VAL_FRACTION"
print("Split verificado: coincide con el usado en M1.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Total: 1320 | train: 1119 | val: 201
Split verificado: coincide con el usado en M1.


## (Correccion) Subir el adaptador LoRA a Hugging Face Hub

Para que el adaptador sea verificable sin necesitar acceso al Drive privado
del proyecto (feedback de revision docente). No depende del resto del
pipeline -- solo necesita que Drive este montado (celda anterior).

Requiere un token de HF con permiso de escritura (`huggingface.co/settings/tokens`,
gratis), guardado en Colab Secrets como `HF_TOKEN`.


In [10]:
from google.colab import userdata
from huggingface_hub import HfApi, login as hf_login

hf_login(token=userdata.get("HF_TOKEN"))

HF_REPO_ID = "Tomas0626/amparo-lora-qwen2.5-7b"

api = HfApi()
api.create_repo(repo_id=HF_REPO_ID, repo_type="model", exist_ok=True, private=False)
api.upload_folder(folder_path=config.DRIVE_ADAPTER_DIR, repo_id=HF_REPO_ID, repo_type="model")
print(f"Adaptador subido: https://huggingface.co/{HF_REPO_ID}")


No files have been modified since last commit. Skipping to prevent empty commit.


Adaptador subido: https://huggingface.co/Tomas0626/amparo-lora-qwen2.5-7b


## (Correccion) Cargar el eval set propio (gold + adversariales)

`data/eval_set.json`: 10 ejemplos gold escritos a mano (no son parte del
dataset de entrenamiento/validacion de M1) mas 3 casos adversariales, cada
uno con un campo `criterio` explicito -- feedback de revision docente sobre
que el eval set de M2 era literalmente el de M1, sin nada adversarial.


In [11]:
from tools.evaluation import eval_set as eval_set_module

eval_records = eval_set_module.load_eval_set()
n_gold = len(eval_set_module.gold_examples(eval_records))
n_adv = len(eval_set_module.adversarial_examples(eval_records))
print(f"Eval set: {len(eval_records)} ejemplos ({n_gold} gold, {n_adv} adversariales)")


Eval set: 13 ejemplos (10 gold, 3 adversariales)


## Fase 1 -- Generacion: baseline (modelo sin fine-tuning)

Puede tardar varios minutos segun el tamano de la validacion (201 ejemplos).


In [12]:
from tools.evaluation import generation

model, tokenizer = generation.load_base_model()

baseline_results = generation.generate_batch(
    model, tokenizer, system_prompt, val_records, label="baseline"
)
print(f"Generadas {len(baseline_results)} respuestas baseline.")


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

[baseline] 10/201 (5%) -- 11.5s/ejemplo, ETA ~36.6 min
[baseline] 20/201 (10%) -- 11.6s/ejemplo, ETA ~34.9 min
[baseline] 30/201 (15%) -- 12.3s/ejemplo, ETA ~35.0 min
[baseline] 40/201 (20%) -- 12.2s/ejemplo, ETA ~32.7 min
[baseline] 50/201 (25%) -- 11.8s/ejemplo, ETA ~29.7 min
[baseline] 60/201 (30%) -- 11.9s/ejemplo, ETA ~27.9 min
[baseline] 70/201 (35%) -- 11.8s/ejemplo, ETA ~25.8 min
[baseline] 80/201 (40%) -- 11.6s/ejemplo, ETA ~23.4 min
[baseline] 90/201 (45%) -- 11.7s/ejemplo, ETA ~21.7 min
[baseline] 100/201 (50%) -- 11.8s/ejemplo, ETA ~19.9 min
[baseline] 110/201 (55%) -- 11.9s/ejemplo, ETA ~18.0 min
[baseline] 120/201 (60%) -- 11.8s/ejemplo, ETA ~16.0 min
[baseline] 130/201 (65%) -- 11.8s/ejemplo, ETA ~14.0 min
[baseline] 140/201 (70%) -- 12.0s/ejemplo, ETA ~12.2 min
[baseline] 150/201 (75%) -- 12.1s/ejemplo, ETA ~10.3 min
[baseline] 160/201 (80%) -- 12.0s/ejemplo, ETA ~8.2 min
[baseline] 170/201 (85%) -- 12.0s/ejemplo, ETA ~6.2 min
[baseline] 180/201 (90%) -- 12.1s/ejemplo, 

In [13]:
eval_set_baseline_results = generation.generate_batch(
    model, tokenizer, system_prompt, eval_records, label="baseline"
)
print(f"Eval set (baseline): {len(eval_set_baseline_results)} respuestas generadas.")


[baseline] 10/13 (77%) -- 13.7s/ejemplo, ETA ~0.7 min
[baseline] 13/13 (100%) -- 12.6s/ejemplo, ETA ~0.0 min
Eval set (baseline): 13 respuestas generadas.


## Fase 2 -- Generacion: fine-tuned (adaptador LoRA desde Drive)


In [14]:
model = generation.attach_adapter(model, config.DRIVE_ADAPTER_DIR)
model.eval()

finetuned_results = generation.generate_batch(
    model, tokenizer, system_prompt, val_records, label="fine_tuned"
)
print(f"Generadas {len(finetuned_results)} respuestas fine-tuned.")


[fine_tuned] 10/201 (5%) -- 4.5s/ejemplo, ETA ~14.4 min
[fine_tuned] 20/201 (10%) -- 4.4s/ejemplo, ETA ~13.3 min
[fine_tuned] 30/201 (15%) -- 4.4s/ejemplo, ETA ~12.5 min
[fine_tuned] 40/201 (20%) -- 4.4s/ejemplo, ETA ~11.8 min
[fine_tuned] 50/201 (25%) -- 4.4s/ejemplo, ETA ~11.0 min
[fine_tuned] 60/201 (30%) -- 4.3s/ejemplo, ETA ~10.2 min
[fine_tuned] 70/201 (35%) -- 4.3s/ejemplo, ETA ~9.4 min
[fine_tuned] 80/201 (40%) -- 4.3s/ejemplo, ETA ~8.7 min
[fine_tuned] 90/201 (45%) -- 4.3s/ejemplo, ETA ~7.9 min
[fine_tuned] 100/201 (50%) -- 4.3s/ejemplo, ETA ~7.2 min
[fine_tuned] 110/201 (55%) -- 4.3s/ejemplo, ETA ~6.5 min
[fine_tuned] 120/201 (60%) -- 4.3s/ejemplo, ETA ~5.8 min
[fine_tuned] 130/201 (65%) -- 4.3s/ejemplo, ETA ~5.1 min
[fine_tuned] 140/201 (70%) -- 4.3s/ejemplo, ETA ~4.4 min
[fine_tuned] 150/201 (75%) -- 4.3s/ejemplo, ETA ~3.7 min
[fine_tuned] 160/201 (80%) -- 4.3s/ejemplo, ETA ~2.9 min
[fine_tuned] 170/201 (85%) -- 4.3s/ejemplo, ETA ~2.2 min
[fine_tuned] 180/201 (90%) -- 4.3s/

In [15]:
eval_set_finetuned_results = generation.generate_batch(
    model, tokenizer, system_prompt, eval_records, label="fine_tuned"
)
print(f"Eval set (fine-tuned): {len(eval_set_finetuned_results)} respuestas generadas.")


[fine_tuned] 10/13 (77%) -- 4.4s/ejemplo, ETA ~0.2 min
[fine_tuned] 13/13 (100%) -- 4.5s/ejemplo, ETA ~0.0 min
Eval set (fine-tuned): 13 respuestas generadas.


## Fase 3 -- Liberar el adaptador y reutilizar el modelo base como juez

`detach_adapter` usa `model.unload()` (quita las capas LoRA sin fusionarlas)
-- así el mismo modelo cargado en memoria sirve como juez independiente del
adaptador, sin una segunda carga completa de modelo.


In [16]:
import torch

model = generation.detach_adapter(model)
torch.cuda.empty_cache()
print("Adaptador liberado. El modelo en memoria es ahora el juez (base, sin fine-tuning).")


Adaptador liberado. El modelo en memoria es ahora el juez (base, sin fine-tuning).


## Fase 4 -- LLM-as-a-Judge

Puntuacion absoluta (1-5 por criterio) de cada respuesta contra la
referencia del dataset, mas un sub-experimento de *position bias* sobre una
muestra de 30 pares baseline/fine-tuned.


In [17]:
from tools.evaluation import judge

judge_baseline = judge.score_batch(model, tokenizer, baseline_results)
judge_finetuned = judge.score_batch(model, tokenizer, finetuned_results)

n_fail_base = sum(1 for s in judge_baseline if not s.parse_ok)
n_fail_ft = sum(1 for s in judge_finetuned if not s.parse_ok)
print(f"Juez baseline: {len(judge_baseline)} filas, {n_fail_base} fallos de parseo.")
print(f"Juez fine-tuned: {len(judge_finetuned)} filas, {n_fail_ft} fallos de parseo.")


[judge] 20/201 (10%) -- 3.3s/ejemplo, ETA ~9.9 min
[judge] 40/201 (20%) -- 3.4s/ejemplo, ETA ~9.2 min
[judge] 60/201 (30%) -- 3.4s/ejemplo, ETA ~8.0 min
[judge] 80/201 (40%) -- 3.4s/ejemplo, ETA ~6.9 min
[judge] 100/201 (50%) -- 3.4s/ejemplo, ETA ~5.8 min
[judge] 120/201 (60%) -- 3.4s/ejemplo, ETA ~4.6 min
[judge] 140/201 (70%) -- 3.4s/ejemplo, ETA ~3.5 min
[judge] 160/201 (80%) -- 3.4s/ejemplo, ETA ~2.3 min
[judge] 180/201 (90%) -- 3.4s/ejemplo, ETA ~1.2 min
[judge] 200/201 (100%) -- 3.4s/ejemplo, ETA ~0.1 min
[judge] 201/201 (100%) -- 3.4s/ejemplo, ETA ~0.0 min
[judge] 20/201 (10%) -- 3.5s/ejemplo, ETA ~10.4 min
[judge] 40/201 (20%) -- 3.4s/ejemplo, ETA ~9.2 min
[judge] 60/201 (30%) -- 3.4s/ejemplo, ETA ~8.0 min
[judge] 80/201 (40%) -- 3.4s/ejemplo, ETA ~6.9 min
[judge] 100/201 (50%) -- 3.4s/ejemplo, ETA ~5.7 min
[judge] 120/201 (60%) -- 3.4s/ejemplo, ETA ~4.6 min
[judge] 140/201 (70%) -- 3.4s/ejemplo, ETA ~3.5 min
[judge] 160/201 (80%) -- 3.4s/ejemplo, ETA ~2.3 min
[judge] 180/201 (

In [18]:
from tools.evaluation import bias

pairs = [
    (b.id, b.query, b.generated, f.generated)
    for b, f in zip(baseline_results, finetuned_results)
]
position_bias_report = bias.run_position_bias_probe(model, tokenizer, pairs)
print(position_bias_report)


[position_bias] 5/30 (17%) -- 2.1s/par, ETA ~0.9 min
[position_bias] 10/30 (33%) -- 2.1s/par, ETA ~0.7 min
[position_bias] 15/30 (50%) -- 2.1s/par, ETA ~0.5 min
[position_bias] 20/30 (67%) -- 2.1s/par, ETA ~0.4 min
[position_bias] 25/30 (83%) -- 2.1s/par, ETA ~0.2 min
[position_bias] 30/30 (100%) -- 2.1s/par, ETA ~0.0 min
PositionBiasReport(n_pairs=30, n_flipped=0, n_tied_or_unparsed=0, flip_rate_pct=0.0, details=[{'id': 554, 'verdict_normal': 'baseline', 'verdict_swapped': 'baseline'}, {'id': 949, 'verdict_normal': 'baseline', 'verdict_swapped': 'baseline'}, {'id': 402, 'verdict_normal': 'baseline', 'verdict_swapped': 'baseline'}, {'id': 1061, 'verdict_normal': 'baseline', 'verdict_swapped': 'baseline'}, {'id': 886, 'verdict_normal': 'baseline', 'verdict_swapped': 'baseline'}, {'id': 1147, 'verdict_normal': 'baseline', 'verdict_swapped': 'baseline'}, {'id': 144, 'verdict_normal': 'baseline', 'verdict_swapped': 'baseline'}, {'id': 1297, 'verdict_normal': 'baseline', 'verdict_swapped': 

## Fase 5 -- Metricas clasicas y metrica de dominio juridico

No requieren GPU (BERTScore descarga un modelo aparte, chico, en CPU/GPU
segun disponibilidad).


In [19]:
from tools.evaluation import pipeline

eval_rows_baseline = pipeline.build_eval_rows(baseline_results, judge_baseline)
eval_rows_finetuned = pipeline.build_eval_rows(finetuned_results, judge_finetuned)

pipeline.fill_bertscore(eval_rows_baseline)
pipeline.fill_bertscore(eval_rows_finetuned)

all_rows = eval_rows_baseline + eval_rows_finetuned
print(f"Total de filas evaluadas: {len(all_rows)}")


config.json:   0%|          | 0.00/648 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/364 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/242k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/480k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/134 [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: dccuchile/bert-base-spanish-wwm-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
pooler.dense.bias                          | MISSING    | 
pooler.dense.weight                        | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Total de filas evaluadas: 402


## Fase 6 -- Sesgos: length bias y self-preference bias


In [20]:
length_bias_baseline = bias.length_bias_correlation(
    [r.judge_composite for r in eval_rows_baseline if r.judge_composite is not None],
    [len(r.generated) for r in eval_rows_baseline if r.judge_composite is not None],
)
length_bias_finetuned = bias.length_bias_correlation(
    [r.judge_composite for r in eval_rows_finetuned if r.judge_composite is not None],
    [len(r.generated) for r in eval_rows_finetuned if r.judge_composite is not None],
)

self_pref_report = bias.self_preference_gap(
    judge_baseline=[r.judge_composite for r in eval_rows_baseline if r.judge_composite is not None],
    judge_finetuned=[r.judge_composite for r in eval_rows_finetuned if r.judge_composite is not None],
    sim_baseline=[r.similarity_pct for r in eval_rows_baseline if r.similarity_pct is not None],
    sim_finetuned=[r.similarity_pct for r in eval_rows_finetuned if r.similarity_pct is not None],
)

bias_summary = {
    "length_bias_pearson_r_baseline": length_bias_baseline["pearson_r"],
    "length_bias_pearson_r_fine_tuned": length_bias_finetuned["pearson_r"],
    "self_preference_judge_gap": self_pref_report.judge_gap,
    "self_preference_similarity_gap": self_pref_report.similarity_gap,
    "self_preference_divergence": self_pref_report.divergence,
    "self_preference_flagged": self_pref_report.flagged,
    "position_bias_flip_rate_pct": position_bias_report.flip_rate_pct,
    "position_bias_n_pairs": position_bias_report.n_pairs,
}
bias_summary


{'length_bias_pearson_r_baseline': -0.411,
 'length_bias_pearson_r_fine_tuned': 0.042,
 'self_preference_judge_gap': 0.063,
 'self_preference_similarity_gap': 0.151,
 'self_preference_divergence': -0.088,
 'self_preference_flagged': False,
 'position_bias_flip_rate_pct': 0.0,
 'position_bias_n_pairs': 30}

## Fase 7 -- Scorecard y persistencia en Drive

Cada corrida se guarda en una carpeta con timestamp propio (no se sobreescribe
la corrida anterior mientras se itera).


In [21]:
import json as _json
import subprocess
from datetime import datetime, timezone
from pathlib import Path

from tools.evaluation import scorecard

manifest = pipeline.build_manifest(n_val=len(val_records), hardware=subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
    capture_output=True, text=True,
).stdout.strip())

run_id = datetime.now(timezone.utc).strftime("%Y-%m-%d_%H%M%S")
drive_run_dir = Path(f"{config.DRIVE_EVAL_OUTPUT_ROOT}/{run_id}")
drive_run_dir.mkdir(parents=True, exist_ok=True)

summaries = scorecard.summarize_by_label(all_rows)
category_summaries = {
    "baseline": scorecard.summarize_by_category(all_rows, "baseline"),
    "fine_tuned": scorecard.summarize_by_category(all_rows, "fine_tuned"),
}
narrative = scorecard.build_narrative(summaries, bias_summary, n_val=len(val_records))

scorecard.export_markdown(
    drive_run_dir / "scorecard.md", summaries, category_summaries, narrative,
    bias_summary, manifest.__dict__,
)
scorecard.export_csv(drive_run_dir / "metricas_por_registro.csv", all_rows)

(drive_run_dir / "run_manifest.json").write_text(
    _json.dumps(manifest.__dict__, indent=2, ensure_ascii=False), encoding="utf-8"
)
(drive_run_dir / "resultados_baseline.jsonl").write_text(
    "\n".join(_json.dumps(r.__dict__, ensure_ascii=False) for r in baseline_results),
    encoding="utf-8",
)
(drive_run_dir / "resultados_finetuned.jsonl").write_text(
    "\n".join(_json.dumps(r.__dict__, ensure_ascii=False) for r in finetuned_results),
    encoding="utf-8",
)
(drive_run_dir / "position_bias_probe.json").write_text(
    _json.dumps(position_bias_report.__dict__, indent=2, ensure_ascii=False),
    encoding="utf-8",
)
(drive_run_dir / "eval_set_baseline_results.jsonl").write_text(
    "\n".join(_json.dumps(r.__dict__, ensure_ascii=False) for r in eval_set_baseline_results),
    encoding="utf-8",
)
(drive_run_dir / "eval_set_finetuned_results.jsonl").write_text(
    "\n".join(_json.dumps(r.__dict__, ensure_ascii=False) for r in eval_set_finetuned_results),
    encoding="utf-8",
)

print(f"Resultados guardados en: {drive_run_dir}")
print((drive_run_dir / "scorecard.md").read_text(encoding="utf-8"))


Resultados guardados en: /content/drive/MyDrive/Colab Notebooks/Amparo/evaluacion/2026-09-19_010048
# Scorecard M2 — Evaluación del Modelo

Generado: 2026-09-19T01:00:48.124583+00:00 · commit `c0d4afd818831cc658ae1ead1516be5ef5089f57` · seed 42 · hardware: NVIDIA L4, 23034 MiB

## Resumen por modelo

| Modelo | N | Exact Match | F1 | BLEU | ROUGE-L | BERTScore | Similitud (%) | Juez (1-5) | Cumplimiento citas (%) | Latencia (s) |
|---|---|---|---|---|---|---|---|---|---|---|
| baseline | 201 | 0.0% | 0.177 | 1.691 | 0.122 | 69.742 | 3.411 | 4.024 | 73.6% | 12.199 |
| fine_tuned | 201 | 0.0% | 0.318 | 8.638 | 0.246 | 78.781 | 18.54 | 4.277 | 100.0% | 4.296 |

## Resumen por categoría

### baseline

| Categoría | N | Similitud (%) | Juez (1-5) | Cumplimiento citas (%) |
|---|---|---|---|---|
| Acceso a informacion publica | 8 | 2.65 | 4.125 | 37.5% |
| Accidentes de transito | 9 | 3.567 | 4.194 | 88.9% |
| Arriendo | 9 | 2.978 | 3.694 | 55.6% |
| Comparendos de transito | 9 | 3.167 | 3.9

## (Opcional) Explorar resultados en pandas


In [22]:
import pandas as pd

df = pd.DataFrame([r.__dict__ for r in all_rows])
df.groupby("label")[["similarity_pct", "judge_composite", "citation_count"]].mean()


,similarity_pct,judge_composite,citation_count
label,,,
baseline,3.411443,4.024112,0.368159
fine_tuned,18.539801,4.277363,0.000000


## Fase 8 -- Juez externo (Groq) -- contraste de auto-preferencia

**Se corre DESPUES de la Fase 7** a proposito: todo lo que necesita la GPU
(generacion, juez Qwen, scorecard) ya quedo guardado en Drive antes de
llegar aqui, asi que si Groq falla o se acaba el cupo gratuito, no se
pierde nada del trabajo caro -- solo hay que reintentar esta seccion
despues (incluso sin GPU, ver `tools/evaluation/run_external_judge_local.py`
para correrla desde tu propia maquina con los archivos ya guardados en
Drive).

El sondeo de position bias con el juez Qwen (Fase 4) encontro que prefirio
la respuesta baseline en los 30 pares, en los dos ordenes -- evidencia de
auto-preferencia (el juez es el mismo modelo base que el baseline). Esta
fase repite el mismo sondeo con un juez de **otra familia** (Groq) sobre
los MISMOS 30 pares (mismo seed), para ver si el patron se sostiene con un
juez independiente. Tambien califica el eval set propio (gold +
adversariales) con ambos jueces.

Requiere `GROQ_API_KEY` en Colab Secrets (gratis, console.groq.com/keys).
La capa gratuita de Groq tiene un limite de ~200,000 tokens/dia por cuenta
-- esta seccion (sondeo + eval set, ~86 llamadas cortas) esta pensada para
caber comodamente en ese cupo. La puntuacion absoluta completa sobre los
201 ejemplos (~400 llamadas) queda aparte, al final, como paso opcional.


In [23]:
import os
from google.colab import userdata

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

from tools.evaluation import external_judge


In [24]:
CHECKPOINT_DIR = Path(f"{config.DRIVE_EVAL_OUTPUT_ROOT}/checkpoints")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

groq_position_bias_report = external_judge.run_position_bias_probe(
    pairs, checkpoint_path=CHECKPOINT_DIR / "groq_position_bias.jsonl"
)
print(groq_position_bias_report)
print("Conteo de veredictos (Groq):", groq_position_bias_report.winner_counts())
print("Conteo de veredictos (Qwen):", position_bias_report.winner_counts())


[external_judge position_bias] 5/30 (17%) -- 0.9s/par, ETA ~0.4 min
[external_judge position_bias] 10/30 (33%) -- 1.4s/par, ETA ~0.5 min
[external_judge position_bias] 15/30 (50%) -- 3.5s/par, ETA ~0.9 min
[external_judge position_bias] 20/30 (67%) -- 4.4s/par, ETA ~0.7 min
[external_judge position_bias] 25/30 (83%) -- 4.9s/par, ETA ~0.4 min
[external_judge position_bias] 30/30 (100%) -- 5.3s/par, ETA ~0.0 min
PositionBiasReport(n_pairs=30, n_flipped=1, n_tied_or_unparsed=0, flip_rate_pct=3.3, details=[{'id': 554, 'verdict_normal': 'baseline', 'verdict_swapped': 'baseline'}, {'id': 949, 'verdict_normal': 'baseline', 'verdict_swapped': 'baseline'}, {'id': 402, 'verdict_normal': 'baseline', 'verdict_swapped': 'baseline'}, {'id': 1061, 'verdict_normal': 'baseline', 'verdict_swapped': 'baseline'}, {'id': 886, 'verdict_normal': 'baseline', 'verdict_swapped': 'baseline'}, {'id': 1147, 'verdict_normal': 'baseline', 'verdict_swapped': 'fine_tuned'}, {'id': 144, 'verdict_normal': 'baseline', 'v

In [25]:
judge_eval_baseline = judge.score_batch(
    model, tokenizer, eval_set_baseline_results,
    checkpoint_path=CHECKPOINT_DIR / "qwen_eval_set_baseline.jsonl",
)
judge_eval_finetuned = judge.score_batch(
    model, tokenizer, eval_set_finetuned_results,
    checkpoint_path=CHECKPOINT_DIR / "qwen_eval_set_finetuned.jsonl",
)

groq_judge_eval_baseline = external_judge.score_batch(
    eval_set_baseline_results,
    checkpoint_path=CHECKPOINT_DIR / "groq_eval_set_baseline.jsonl",
)
groq_judge_eval_finetuned = external_judge.score_batch(
    eval_set_finetuned_results,
    checkpoint_path=CHECKPOINT_DIR / "groq_eval_set_finetuned.jsonl",
)

print("=== Eval set: respuestas y puntuaciones (revisar manualmente contra 'criterio') ===")
for rec, gen_b, gen_f, jb, jf, gb, gf in zip(
    eval_records, eval_set_baseline_results, eval_set_finetuned_results,
    judge_eval_baseline, judge_eval_finetuned, groq_judge_eval_baseline, groq_judge_eval_finetuned,
):
    print(f"\n--- id {rec['id']} ({rec['tipo']}) {rec['category']} ---")
    print(f"Pregunta: {rec['messages'][1]['content']}")
    print(f"Criterio: {rec['criterio']}")
    print(f"[baseline]   {gen_b.generated}")
    print(f"  Qwen: {jb.composite}  Groq: {gb.composite}")
    print(f"[fine-tuned] {gen_f.generated}")
    print(f"  Qwen: {jf.composite}  Groq: {gf.composite}")


[judge] 13/13 (100%) -- 3.5s/ejemplo, ETA ~0.0 min
[judge] 13/13 (100%) -- 3.6s/ejemplo, ETA ~0.0 min
[external_judge] 5/13 (38%) -- 0.6s/ejemplo, ETA ~0.1 min
[external_judge] 10/13 (77%) -- 0.6s/ejemplo, ETA ~0.0 min
[external_judge] 13/13 (100%) -- 2.0s/ejemplo, ETA ~0.0 min
[external_judge] 5/13 (38%) -- 4.7s/ejemplo, ETA ~0.6 min
[external_judge] 10/13 (77%) -- 4.8s/ejemplo, ETA ~0.2 min
[external_judge] 13/13 (100%) -- 5.0s/ejemplo, ETA ~0.0 min
=== Eval set: respuestas y puntuaciones (revisar manualmente contra 'criterio') ===

--- id 9001 (gold) Arriendo ---
Pregunta: Mi arrendador me subio el canon de arrendamiento el doble de lo que pagaba el año pasado, ¿puede hacer eso?
Criterio: Debe mencionar que el incremento tiene un limite legal (referenciado al IPC o similar) sin citar el numero de ley/decreto; debe orientar a revisar el contrato y reclamar, no afirmar un porcentaje exacto con falsa certeza.
[baseline]   En Colombia, la ley del arrendamiento (Ley 100 de 1993) establec

## Fase 8b -- Persistir resultados de Groq (sondeo + eval set)


In [26]:
groq_dir = drive_run_dir

(groq_dir / "groq_position_bias_probe.json").write_text(
    _json.dumps(groq_position_bias_report.__dict__, indent=2, ensure_ascii=False),
    encoding="utf-8",
)
(groq_dir / "eval_set_resultados.jsonl").write_text(
    "\n".join(
        _json.dumps({
            "id": rec["id"],
            "tipo": rec["tipo"],
            "category": rec["category"],
            "criterio": rec["criterio"],
            "query": rec["messages"][1]["content"],
            "baseline_generated": gb.generated,
            "finetuned_generated": gf_.generated,
            "qwen_judge_baseline": jb.composite,
            "qwen_judge_finetuned": jf.composite,
            "groq_judge_baseline": gjb.composite,
            "groq_judge_finetuned": gjf.composite,
        }, ensure_ascii=False)
        for rec, gb, gf_, jb, jf, gjb, gjf in zip(
            eval_records, eval_set_baseline_results, eval_set_finetuned_results,
            judge_eval_baseline, judge_eval_finetuned,
            groq_judge_eval_baseline, groq_judge_eval_finetuned,
        )
    ),
    encoding="utf-8",
)
print(f"Resultados de Groq (position bias + eval set) guardados en: {groq_dir}")


Resultados de Groq (position bias + eval set) guardados en: /content/drive/MyDrive/Colab Notebooks/Amparo/evaluacion/2026-09-19_010048


## (Opcional, consume la mayoria del cupo gratuito diario) Puntuacion absoluta completa con Groq

Esto califica los 201 ejemplos de validacion x2 (baseline y fine-tuned) con
Groq -- unas 400 llamadas, suficiente para agotar el cupo gratuito diario
por si solo (ya paso una vez). El hallazgo principal de auto-preferencia
(el 30/30 de la Fase 8) **no depende de esta celda** -- sáltatela si no te
sobra cupo, o córrela otro dia con cupo fresco.


In [27]:
from tools.evaluation import external_judge
import statistics

groq_judge_baseline = external_judge.score_batch(
    baseline_results, checkpoint_path=CHECKPOINT_DIR / "groq_absolute_baseline.jsonl"
)
groq_judge_finetuned = external_judge.score_batch(
    finetuned_results, checkpoint_path=CHECKPOINT_DIR / "groq_absolute_finetuned.jsonl"
)

n_fail_base_groq = sum(1 for s in groq_judge_baseline if not s.parse_ok)
n_fail_ft_groq = sum(1 for s in groq_judge_finetuned if not s.parse_ok)
print(f"Groq baseline: {len(groq_judge_baseline)} filas, {n_fail_base_groq} fallos de parseo.")
print(f"Groq fine-tuned: {len(groq_judge_finetuned)} filas, {n_fail_ft_groq} fallos de parseo.")

avg_groq_base = statistics.fmean(s.composite for s in groq_judge_baseline if s.composite is not None)
avg_groq_ft = statistics.fmean(s.composite for s in groq_judge_finetuned if s.composite is not None)
print(f"Groq juez (compuesto 1-5): baseline {avg_groq_base:.3f}, fine-tuned {avg_groq_ft:.3f}")


[external_judge] 5/201 (2%) -- 6.4s/ejemplo, ETA ~21.0 min
[external_judge] 10/201 (5%) -- 6.2s/ejemplo, ETA ~19.7 min
[external_judge] 15/201 (7%) -- 6.0s/ejemplo, ETA ~18.7 min
[external_judge] 20/201 (10%) -- 6.0s/ejemplo, ETA ~18.2 min
[external_judge] 25/201 (12%) -- 6.1s/ejemplo, ETA ~18.0 min
[external_judge] 30/201 (15%) -- 6.1s/ejemplo, ETA ~17.3 min
[external_judge] 35/201 (17%) -- 6.1s/ejemplo, ETA ~16.9 min
[external_judge] 40/201 (20%) -- 6.1s/ejemplo, ETA ~16.4 min
[external_judge] 45/201 (22%) -- 6.1s/ejemplo, ETA ~15.8 min
[external_judge] 50/201 (25%) -- 6.0s/ejemplo, ETA ~15.2 min
[external_judge] 55/201 (27%) -- 6.1s/ejemplo, ETA ~14.7 min
[external_judge] 60/201 (30%) -- 6.1s/ejemplo, ETA ~14.3 min
[external_judge] 65/201 (32%) -- 6.1s/ejemplo, ETA ~13.7 min
[external_judge] 70/201 (35%) -- 6.1s/ejemplo, ETA ~13.2 min
[external_judge] 75/201 (37%) -- 6.0s/ejemplo, ETA ~12.7 min
[external_judge] 80/201 (40%) -- 6.0s/ejemplo, ETA ~12.2 min
[external_judge] 85/201 (42%

StatisticsError: fmean requires at least one data point

## (Opcional) Persistir la puntuacion absoluta completa de Groq

Solo corre esto si corriste la celda opcional de arriba.


In [ ]:
(groq_dir / "groq_judge_baseline.jsonl").write_text(
    "\n".join(_json.dumps(s.__dict__, ensure_ascii=False) for s in groq_judge_baseline),
    encoding="utf-8",
)
(groq_dir / "groq_judge_finetuned.jsonl").write_text(
    "\n".join(_json.dumps(s.__dict__, ensure_ascii=False) for s in groq_judge_finetuned),
    encoding="utf-8",
)
print(f"Puntuacion absoluta de Groq guardada en: {groq_dir}")
